In [1]:
# ─── Imports ──────────────────────────────────────────────────────────────────
import re
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import ConnectionPatch
import seaborn as sns
import shutil
from pathlib import Path
from collections import defaultdict
from satpy.enhancements.enhancer import get_enhanced_image
from SARS.satellite_data_processor import *
from SARS.sat_info import *
from pathlib import Path
import os


In [2]:
# ─── Paths ────────────────────────────────────────────────────────────────────
data_dir = Path('~/Downloads/sars_p1_data/processed_output').expanduser()
out_dir = Path('~/Downloads/sars_p1_data/plots').expanduser()
img_dir = Path('~/Downloads/sars_p1_data/imgs').expanduser()
stats_dir = Path('~/Downloads/sars_p1_data/stats').expanduser()
latex_dir = Path('~/Downloads/examples-master 2/Latex').expanduser()

for d in [out_dir, img_dir, stats_dir]:
    d.mkdir(parents=True, exist_ok=True)

corrected_data_paths = sorted(data_dir.glob('*_corrected*'))
uncorrected_data_paths = sorted(data_dir.glob('*_unc*'))
all_data_paths = sorted(data_dir.glob('*.nc'))
sentinel2b_files = sorted(data_dir.glob("*sentinel*_cor*"))
terra_files = sorted(data_dir.glob("*terra*.nc"))
noaa20_files = sorted(data_dir.glob("*noaa20*.nc"))


In [3]:
# ─── Band metadata ────────────────────────────────────────────────────────────
MODIS_BANDS = {
    1: {'wavelength': '0.620–0.670', 'primary_use': 'Land/cloud boundaries'},
    2: {'wavelength': '0.841–0.876', 'primary_use': 'Land/cloud boundaries'},
    3: {'wavelength': '0.459–0.479', 'primary_use': 'Land/cloud properties'},
    4: {'wavelength': '0.545–0.565', 'primary_use': 'Land/cloud properties'},
    5: {'wavelength': '1.230–1.250', 'primary_use': 'Land/cloud properties'},
    6: {'wavelength': '1.628–1.652', 'primary_use': 'Land/cloud properties'},
    7: {'wavelength': '2.105–2.155', 'primary_use': 'Land/cloud properties'},
    8: {'wavelength': '0.405–0.420', 'primary_use': 'Ocean color/phytoplankton'},
    9: {'wavelength': '0.438–0.448', 'primary_use': 'Ocean color/phytoplankton'},
    10: {'wavelength': '0.483–0.493', 'primary_use': 'Ocean color/phytoplankton'},
    11: {'wavelength': '0.526–0.536', 'primary_use': 'Ocean color/phytoplankton'},
    12: {'wavelength': '0.546–0.556', 'primary_use': 'Ocean color/phytoplankton'},
    13: {'wavelength': '0.662–0.672', 'primary_use': 'Ocean color/phytoplankton'},
    14: {'wavelength': '0.673–0.683', 'primary_use': 'Ocean color/phytoplankton'},
    15: {'wavelength': '0.743–0.753', 'primary_use': 'Ocean color/phytoplankton'},
    16: {'wavelength': '0.862–0.877', 'primary_use': 'Ocean color/phytoplankton'},
    17: {'wavelength': '0.890–0.920', 'primary_use': 'Atmospheric water vapor'},
    18: {'wavelength': '0.931–0.941', 'primary_use': 'Atmospheric water vapor'},
    19: {'wavelength': '0.915–0.965', 'primary_use': 'Atmospheric water vapor'},
    20: {'wavelength': '3.660–3.840', 'primary_use': 'Surface/cloud temperature'},
    21: {'wavelength': '3.929–3.989', 'primary_use': 'Surface/cloud temperature'},
    22: {'wavelength': '3.929–3.989', 'primary_use': 'Surface/cloud temperature'},
    23: {'wavelength': '4.020–4.080', 'primary_use': 'Surface/cloud temperature'},
    24: {'wavelength': '4.433–4.498', 'primary_use': 'Atmospheric temperature'},
    25: {'wavelength': '4.482–4.549', 'primary_use': 'Atmospheric temperature'},
    26: {'wavelength': '1.360–1.390', 'primary_use': 'Cirrus cloud detection'},
    27: {'wavelength': '6.535–6.895', 'primary_use': 'Mid-troposphere water vapor'},
    28: {'wavelength': '7.175–7.475', 'primary_use': 'Upper-troposphere water vapor'},
    29: {'wavelength': '8.400–8.700', 'primary_use': 'Surface/cloud temperature'},
    30: {'wavelength': '9.580–9.880', 'primary_use': 'Total ozone'},
    31: {'wavelength': '10.780–11.280', 'primary_use': 'Surface/cloud temperature'},
    32: {'wavelength': '11.770–12.270', 'primary_use': 'Surface/cloud temperature'},
    33: {'wavelength': '13.185–13.485', 'primary_use': 'Cloud top altitude'},
    34: {'wavelength': '13.485–13.785', 'primary_use': 'Cloud top altitude'},
    35: {'wavelength': '13.785–14.085', 'primary_use': 'Cloud top altitude'},
    36: {'wavelength': '14.085–14.385', 'primary_use': 'Cloud top altitude'},
}
NOAA20_VIIRS_BANDS = {
    'M1': {'wavelength': '0.402–0.422', 'primary_use': 'Ocean color/aerosol'},
    'M2': {'wavelength': '0.436–0.454', 'primary_use': 'Ocean color/aerosol'},
    'M3': {'wavelength': '0.478–0.498', 'primary_use': 'Ocean color/aerosol'},
    'M4': {'wavelength': '0.545–0.565', 'primary_use': 'Ocean color/green vegetation'},
    'M5': {'wavelength': '0.662–0.682', 'primary_use': 'Ocean color/vegetation'},
    'M6': {'wavelength': '0.739–0.754', 'primary_use': 'Atmospheric correction'},
    'M7': {'wavelength': '0.846–0.885', 'primary_use': 'Ocean color/vegetation/cloud'},
    'M8': {'wavelength': '1.230–1.250', 'primary_use': 'Cloud particle size'},
    'M9': {'wavelength': '1.371–1.386', 'primary_use': 'Cirrus cloud detection'},
    'M10': {'wavelength': '1.580–1.640', 'primary_use': 'Snow/ice discrimination'},
    'M11': {'wavelength': '2.225–2.275', 'primary_use': 'Cloud/snow/fire detection'},
    'M12': {'wavelength': '3.610–3.790', 'primary_use': 'Sea surface temperature'},
    'M13': {'wavelength': '3.973–4.128', 'primary_use': 'Sea surface temp/fire detection'},
    'M14': {'wavelength': '8.400–8.700', 'primary_use': 'Cloud top properties'},
    'M15': {'wavelength': '10.263–11.263', 'primary_use': 'Sea surface temperature'},
    'M16': {'wavelength': '11.538–12.488', 'primary_use': 'Sea surface temperature'},
    'I1': {'wavelength': '0.600–0.680', 'primary_use': 'Imagery/land surface'},
    'I2': {'wavelength': '0.846–0.885', 'primary_use': 'Imagery/NDVI'},
    'I3': {'wavelength': '1.580–1.640', 'primary_use': 'Imagery/binary snow cover'},
    'I4': {'wavelength': '3.550–3.930', 'primary_use': 'Imagery/fire detection'},
    'I5': {'wavelength': '10.500–12.400', 'primary_use': 'Imagery/surface temperature'},
}
SENTINEL2B_BANDS = {
    'B01': {'wavelength': '0.433–0.453', 'primary_use': 'Coastal aerosol'},
    'B02': {'wavelength': '0.458–0.523', 'primary_use': 'Blue/aerosol correction'},
    'B03': {'wavelength': '0.543–0.578', 'primary_use': 'Green vegetation'},
    'B04': {'wavelength': '0.650–0.680', 'primary_use': 'Red/vegetation chlorophyll'},
    'B05': {'wavelength': '0.698–0.713', 'primary_use': 'Vegetation red edge'},
    'B06': {'wavelength': '0.733–0.748', 'primary_use': 'Vegetation red edge'},
    'B07': {'wavelength': '0.765–0.785', 'primary_use': 'Vegetation red edge/LAI'},
    'B08': {'wavelength': '0.785–0.900', 'primary_use': 'NIR/vegetation'},
    'B8A': {'wavelength': '0.855–0.875', 'primary_use': 'NIR narrow/vegetation'},
    'B09': {'wavelength': '0.935–0.955', 'primary_use': 'Water vapor'},
    'B10': {'wavelength': '1.360–1.390', 'primary_use': 'Cirrus cloud detection'},
    'B11': {'wavelength': '1.565–1.655', 'primary_use': 'SWIR/snow and ice'},
    'B12': {'wavelength': '2.100–2.280', 'primary_use': 'SWIR/geology/soil moisture'},
}
SAT_BAND_METADATA = {
    'terra': MODIS_BANDS,
    'aqua': MODIS_BANDS,
    'noaa20': NOAA20_VIIRS_BANDS,
    'sentinel2b': SENTINEL2B_BANDS,
}

# ─── Constants ────────────────────────────────────────────────────────────────
CLASS_COLORS = {
    'inland_water': '#2196F3',
    'snow': '#90CAF9',
    'desert': '#D4A257',
    'agriculture': '#66BB6A',
    'urban': '#78909C',
    'grassland': '#9CCC65',
    'dense_forest': '#2E7D32',
    'dust': '#BCAAA4',
    'smoke': '#616161',
    'high_cloud': '#CE93D8',
}
ATMOSPHERIC_CLASSES = {'dust', 'smoke', 'high_cloud'}
LAND_CLASSES = {'inland_water', 'snow', 'desert', 'agriculture', 'urban', 'grassland', 'dense_forest'}

SAT_COLORS = {
    'terra': '#E53935',
    'aqua': '#1E88E5',
    'noaa20': '#43A047',
    'sentinel2b': '#8E24AA',
}
SAT_STYLES = {
    ('terra', 'corrected'): {'linestyle': '-', 'linewidth': 1},
    ('terra', 'uncorrected'): {'linestyle': '--', 'linewidth': 1},
    ('noaa20', 'corrected'): {'linestyle': '-', 'linewidth': 1},
    ('noaa20', 'uncorrected'): {'linestyle': '--', 'linewidth': 1},
    ('sentinel2b', 'corrected'): {'linestyle': '-', 'linewidth': 1},
    ('sentinel2b', 'uncorrected'): {'linestyle': '--', 'linewidth': 1},
}
SAT_ORBIT_PARAMS = {
    'terra': {'orbit_height': 705_000, 'along_track_res': 1_000, 'cross_track_res': 1_000, 'earth_curvature': True,
              'clean_name': 'Terra MODIS'},
    'aqua': {'orbit_height': 705_000, 'along_track_res': 1_000, 'cross_track_res': 1_000, 'earth_curvature': True,
             'clean_name': 'Aqua MODIS'},
    'noaa20': {'orbit_height': 824_000, 'along_track_res': 750, 'cross_track_res': 750, 'earth_curvature': True,
               'clean_name': 'NOAA 20 VIIRS'},
    'sentinel2b': {'orbit_height': 786_000, 'along_track_res': 60, 'cross_track_res': 60, 'earth_curvature': False,
                   'clean_name': 'Sentinel-2B MSI'},
}


In [4]:

# ─── Helper functions ─────────────────────────────────────────────────────────
def clean_satname(satname):
    return {'noaa20': 'NOAA 20', 'terra': 'Terra', 'aqua': 'Aqua', 'sentinel2b': 'Sentinel-2B'}.get(satname.lower(),
                                                                                                    satname.title())


def clean_class(pixel_class):
    return pixel_class.replace('_', ' ').title()


def clean_cortype(cor_type):
    return cor_type.replace('_', ' ').title()


def make_rect(x0, y0, x1, y1, color, lw=1.5):
    return mpatches.Rectangle(
        (x0 - 0.5, y0 - 0.5), x1 - x0, y1 - y0,
        linewidth=lw, edgecolor=color, facecolor='none'
    )


def find_val(pixel_class, wl_c, data_dict, tol=0.015):
    for wl, val in data_dict.get(pixel_class, {}).items():
        if abs(wl - wl_c) <= tol:
            return val
    return float('nan')


def normalize_to_north_up(ds, satellite_name):
    ASCENDING_SATELLITES = {'noaa20', 'noaa-20'}
    if satellite_name.lower() in ASCENDING_SATELLITES:
        lat = ds['latitude'].values
        lat_top = float(np.nanmean(lat[:5, :]))
        lat_bottom = float(np.nanmean(lat[-5:, :]))
        if lat_top < lat_bottom:
            print(f"[{satellite_name}] Flipping to north-up")
            ds = ds.isel(y=slice(None, None, -1))
    return ds


def get_sat_style(satname, cor_type):
    return {
        'linestyle': '--' if cor_type == 'uncorrected' else '-',
        'linewidth': 1.0,
    }


def plot_spectral_class(ax, x, y, yerr, label, color, linestyle='-', linewidth=2.0):
    ax.plot(x, y, color=color, linewidth=linewidth, linestyle=linestyle, label=label, zorder=3,
            marker='s', markersize=0.8)
    ax.fill_between(x, y - yerr, y + yerr, color=color, alpha=0.15, zorder=2)
    ax.errorbar(
        x, y,
        yerr=yerr,
        fmt='none',
        ecolor=color,
        elinewidth=0.8,
        capsize=3,
        capthick=0.8,
        alpha=0.6,
        zorder=4
    )

def clean_attrs_for_netcdf(ds: xr.Dataset) -> xr.Dataset:
    safe_types = (str, int, float, np.ndarray, np.integer, np.floating, bytes)
    ds = ds.copy()
    ds.attrs = {k: v for k, v in ds.attrs.items() if isinstance(v, safe_types)}
    for var in list(ds.data_vars) + list(ds.coords):
        try:
            ds[var].attrs = {k: v for k, v in ds[var].attrs.items() if isinstance(v, safe_types)}
        except Exception:
            pass
    return ds


# ─── Spectral analysis functions ──────────────────────────────────────────────
def make_class_plots(pixel_class, class_type='reflectance', outputdir=out_dir):
    """Plot all satellites/cor_types for a single pixel class."""
    fig, ax = plt.subplots(figsize=(11, 5))

    for (satname, cor_type), pixel_classes in spectral_analysis.items():
        if pixel_class not in pixel_classes:
            continue

        sa = pixel_classes[pixel_class]
        style = get_sat_style(satname, cor_type)
        color = SAT_COLORS.get(satname, '#333333')
        label = f'{clean_satname(satname)} ({clean_cortype(cor_type)})'

        if class_type == 'reflectance':
            plot_spectral_class(
                ax,
                sa['wl_ref_sorted'], sa['reflectance_mean'], sa['reflectance_stdv'],
                # sa['wl_ref_lower'], sa['wl_ref_upper'],
                label, color, **style
            )
        else:
            plot_spectral_class(
                ax,
                sa['wl_bt_sorted'], sa['brightness_temp_mean'], sa['brightness_temp_stdv'],
                # sa['wl_bt_lower'], sa['wl_bt_upper'],
                label, color, **style
            )

    band_type = 'Reflective' if class_type == 'reflectance' else 'Emissive'

    if class_type == 'reflectance':
        ax.axhspan(100, 140, color='#888888', alpha=0.12, zorder=0)
        ax.axhline(100, color='#888888', linewidth=0.8, linestyle='--', zorder=1)
        ax.set_ylim(0, 140)
        ax.set_xlim(0.25, 2.3)
        ax.set_ylabel('Reflectance (%)', fontsize=11)
    else:
        ax.set_ylim(220, 340)
        ax.set_xlim(3, 15)
        ax.set_ylabel('Brightness Temperature (K)', fontsize=11)

    ax.margins(x=0)
    ax.set_title(f'{clean_class(pixel_class)}  ·  {band_type} Bands  ·  All Satellites',
                 fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel('Wavelength (µm)', fontsize=11)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.,
              framealpha=0.9, edgecolor='#cccccc', fontsize=9)
    sns.despine(ax=ax, left=False, bottom=False)
    fig.tight_layout()
    fig.savefig(outputdir / f'{pixel_class}_{band_type}_all_satellites.png', dpi=150, bbox_inches='tight')
    plt.show()


def make_band_table(satname, cor_type='corrected'):
    """Single table per satellite combining reflective and emissive bands."""
    band_meta = SAT_BAND_METADATA.get(satname)
    if band_meta is None:
        print(f"No band metadata for {satname}")
        return None

    sat_data_ref = {}
    sat_data_bt = {}

    for (sn, ct), pixel_classes in spectral_analysis.items():
        if sn != satname or ct != cor_type:
            continue
        for pixel_class, sa in pixel_classes.items():
            sat_data_ref[pixel_class] = dict(zip(sa['wl_ref_sorted'], sa['reflectance_mean']))
            sat_data_bt[pixel_class] = dict(zip(sa['wl_bt_sorted'], sa['brightness_temp_mean']))

    if not sat_data_ref and not sat_data_bt:
        print(f"No {cor_type} data found for {satname}")
        return None

    all_classes = sorted(set(list(sat_data_ref.keys()) + list(sat_data_bt.keys())))

    rows = []
    for band_key, info in band_meta.items():
        lo, hi = [float(x) for x in info['wavelength'].split('–')]
        wl_c = round((lo + hi) / 2, 4)
        is_emissive = lo >= 3.0
        band_type = 'Emissive' if is_emissive else 'Reflective'

        row = {
            'Band': band_key,
            'Wavelength': f"{lo:.3f}–{hi:.3f}",
            'Type': band_type,
        }
        for pixel_class in all_classes:
            data = sat_data_bt if is_emissive else sat_data_ref
            val = find_val(pixel_class, wl_c, data)
            row[clean_class(pixel_class)] = round(val, 3) if not np.isnan(val) else None

        rows.append(row)

    df = pd.DataFrame(rows)
    df['_lo'] = df['Wavelength'].apply(lambda x: float(x.split('–')[0]))
    df = (df[df['Type'] == 'Reflective'].sort_values('_lo')
          ._append(df[df['Type'] == 'Emissive'].sort_values('_lo'))
          .drop(columns='_lo')
          .reset_index(drop=True))
    return df


def make_band_table_pctdiff(satname):
    """Percent difference: (corrected - uncorrected) / uncorrected * 100."""
    df_cor = make_band_table(satname, 'corrected')
    df_uncor = make_band_table(satname, 'uncorrected')

    if df_cor is None or df_uncor is None:
        return None

    classes = [c for c in df_cor.columns if c not in ('Band', 'Wavelength', 'Type')]

    # Check if reflective bands actually differ
    ref_rows = df_cor[df_cor['Type'] == 'Reflective']
    ref_rows2 = df_uncor[df_uncor['Type'] == 'Reflective']
    ref_vals = ref_rows[classes].apply(pd.to_numeric, errors='coerce')
    ref_vals2 = ref_rows2[classes].apply(pd.to_numeric, errors='coerce')
    if not (abs(ref_vals.values - ref_vals2.values) > 0.001).any():
        print(f"[{satname}] No reflective difference — skipping pctdiff table.")
        return None

    df_pct = df_cor[['Band', 'Wavelength', 'Type']].copy()
    for c in classes:
        cor_vals = pd.to_numeric(df_cor[c], errors='coerce')
        uncor_vals = pd.to_numeric(df_uncor[c], errors='coerce')
        df_pct[c] = (100 * (cor_vals - uncor_vals) / uncor_vals).round(2)

    # Drop emissive rows if all zero
    emissive_rows = df_pct[df_pct['Type'] == 'Emissive']
    emissive_vals = emissive_rows[classes].apply(pd.to_numeric, errors='coerce')
    if not (emissive_vals.abs() > 0.001).any().any():
        print(f"[{satname}] Emissive bands identical — reflective only in pctdiff.")
        df_pct = df_pct[df_pct['Type'] == 'Reflective'].reset_index(drop=True)

    return df_pct


def make_summary_with_pctdiff(df_stats, band_type):
    df = (
        df_stats
        .query(f"band_type == '{band_type}'")
        .groupby(['satellite', 'correction_type', 'class'])
        .agg(mean=('mean', 'mean'), stdv=('stdv', 'mean'), n_bands=('wavelength', 'count'))
        .round(3)
        .reset_index()
    )
    found = df['correction_type'].unique()
    if 'Corrected' not in found or 'Uncorrected' not in found:
        print(f"WARNING [{band_type}]: only found: {found}")

    corrected = df[df['correction_type'] == 'Corrected'].set_index(['satellite', 'class'])
    uncorrected = df[df['correction_type'] == 'Uncorrected'].set_index(['satellite', 'class'])

    summary = corrected[['mean', 'stdv', 'n_bands']].copy()
    summary.columns = ['Corrected Mean', 'Corrected Std', 'N Bands']
    summary['Uncorrected Mean'] = uncorrected['mean']
    summary['Uncorrected Std'] = uncorrected['stdv']
    summary['% Difference'] = (
            100 * (summary['Corrected Mean'] - summary['Uncorrected Mean'])
            / summary['Uncorrected Mean']
    ).round(2)
    return summary.reset_index()


# ─── LaTeX helpers ────────────────────────────────────────────────────────────
def summary_to_latex(df, caption, label, clearpage=True):
    lines = []
    lines.append(r'\begin{table}[h!tbp]')
    lines.append(r'\centering')
    lines.append(r'\small')
    lines.append(rf'\caption{{{caption}}}')
    lines.append(rf'\label{{{label}}}')
    lines.append(r'\begin{tabular}{llrrrrrr}')
    lines.append(r'\toprule')
    lines.append(
        r'Satellite & Class & \multicolumn{2}{c}{Corrected} & \multicolumn{2}{c}{Uncorrected} & \% Diff & $N$ \\')
    lines.append(r'\cmidrule(lr){3-4} \cmidrule(lr){5-6}')
    lines.append(r' & & Mean & Std & Mean & Std & & \\')
    lines.append(r'\midrule')

    prev_sat = None
    for _, row in df.iterrows():
        if row['satellite'] != prev_sat and prev_sat is not None:
            lines.append(r'\midrule')
        prev_sat = row['satellite']
        pct = row['% Difference']
        pct_str = rf'\textcolor{{red}}{{{pct:+.2f}}}' if abs(pct) > 5 else f'{pct:+.2f}'
        lines.append(
            f"{row['satellite']} & {row['class']} & "
            f"{row['Corrected Mean']:.3f} & {row['Corrected Std']:.3f} & "
            f"{row['Uncorrected Mean']:.3f} & {row['Uncorrected Std']:.3f} & "
            f"{pct_str} & {int(row['N Bands'])} \\\\"
        )

    lines.append(r'\bottomrule')
    lines.append(
        rf'\multicolumn{{8}}{{l}}{{\footnotesize Reflective values in \%; emissive values in K. Values $>$5\% highlighted.}}\\')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')
    if clearpage:
        lines.append(r'\clearpage')
    return '\n'.join(lines)


def band_df_to_latex(df, satname, caption, label, fontsize=None, is_pctdiff=False, is_stdv=False, clearpage=True):
    classes = [c for c in df.columns if c not in ('Band', 'Wavelength', 'Primary Use', 'Type')]
    n_classes = len(classes)
    n_bands = len(df)

    if fontsize is None:
        fontsize = r'\scriptsize' if (n_bands > 30 or n_classes > 8) else r'\footnotesize'

    band_col = 0.5 if n_bands > 20 else 0.7
    wl_col = 1.6 if n_bands > 20 else 1.8
    tabcolsep = 2 if n_bands > 20 else 3

    fixed_width = band_col + wl_col
    class_width = round((24.0 - fixed_width) / n_classes, 2)

    col_format = (
            f'C{{{band_col}cm}} C{{{wl_col}cm}} '
            + ' '.join([f'C{{{class_width}cm}}'] * n_classes)
    )

    lines = []
    # lines.append(r'\begin{landscape}')
    lines.append(r'\begin{table}[h!tbp]')
    lines.append(r'\centering')
    lines.append(rf'\setlength{{\tabcolsep}}{{3pt}}')
    lines.append(fontsize)
    lines.append(rf'\caption{{{caption}}}')
    lines.append(rf'\label{{{label}}}')
    lines.append(r'\begin{tabular}{C{0.5cm} C{1.6cm} C{1.2cm} C{1.2cm} C{1.2cm} C{1.2cm} C{1.2cm} C{1.2cm} C{1.2cm} C{1.2cm} C{1.2cm} C{1.2cm}}')
    lines.append(r'\toprule')
    lines.append(
        r'\textbf{Band} & \textbf{Wavelength} ($\mu$m) & '
        + ' & '.join(rf'\textbf{{{c}}}' for c in classes) + r' \\'
    )
    lines.append(r'\midrule')

    prev_type = None
    for _, row in df.iterrows():
        if row['Type'] != prev_type and prev_type is not None:
            lines.append(r'\midrule')
        prev_type = row['Type']

        vals = [str(row['Band']), row['Wavelength']]
        for c in classes:
            v = row[c]
            if v is None or (isinstance(v, float) and np.isnan(v)):
                vals.append('--')
            elif is_pctdiff:
                pct = float(v)
                if pct > 20:
                    vals.append(rf'\textcolor{{blue}}{{{pct:+.2f}}}')
                elif pct < -20:
                    vals.append(rf'\textcolor{{red}}{{{pct:+.2f}}}')
                else:
                    vals.append(f'{pct:+.2f}')
            elif is_stdv:
                sd = float(v)
                if sd > 1:
                    vals.append(rf'\textcolor{{red}}{{{sd:+.2f}}}')
                else:
                    vals.append(f'{sd:+.2f}')
            else:
                vals.append(f"{float(v):.3f}")
        lines.append(' & '.join(vals) + r' \\')

    lines.append(r'\bottomrule')
    if is_pctdiff:
        footer = (r'Percent difference (\%) between corrected and uncorrected. '
                  r'\textcolor{blue}{Blue} = increase $>$20\%; \textcolor{red}{Red} = decrease $>$20\%.')
    if is_stdv:
        footer = (r'Standard deviation (\%) of corrected and uncorrected. '
                  r'\textcolor{red}{Red} = $>$1\%.')
    else:
        footer = r'Reflective values in \%; emissive values in K. Corrected only.'
    lines.append(rf'\addlinespace[3pt]')
    lines.append(rf'\multicolumn{{{2 + n_classes}}}{{l}}{{\scriptsize {footer}}}\\')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')
    if clearpage:
        lines.append(r'\clearpage')
    # lines.append(r'\end{landscape}')
    return '\n'.join(lines)


def pixel_info_to_latex(df, caption, label, clearpage=True):
    lines = []
    # lines.append(r'\begin{landscape}')
    lines.append(r'\begin{table}[h!tbp]')
    lines.append(r'\centering')
    lines.append(r'\scriptsize')
    lines.append(r'\setlength{\tabcolsep}{3pt}')
    lines.append(rf'\caption{{{caption}}}')
    lines.append(rf'\label{{{label}}}')
    lines.append(
        r'\begin{tabular}{'
        r'P{2.2cm} P{1.8cm} C{1.3cm} C{1.5cm} '
        r'C{1.2cm} C{1.4cm} C{1.2cm} C{1.4cm} C{1.2cm} C{1.2cm}}'
    )
    lines.append(r'\toprule')
    lines.append(
        r'\textbf{Satellite} & \textbf{Class} & \textbf{Lat} & \textbf{Lon} & '
        r'\textbf{SZA} ($^\circ$) & \textbf{SAA} ($^\circ$) & '
        r'\textbf{VZA} ($^\circ$) & \textbf{VAA} ($^\circ$) & '
        r'$p_\times$ (m) & $p_\parallel$ (m) \\'
    )
    lines.append(r'\midrule')

    prev_sat = None
    for _, row in df.sort_values(['satellite', 'class']).iterrows():
        if row['satellite'] != prev_sat and prev_sat is not None:
            lines.append(r'\midrule')
        prev_sat = row['satellite']
        lines.append(
            f"{row['satellite']} & {row['class']} & "
            f"{row['latitude']:.4f} & {row['longitude']:.4f} & "
            f"{row['sza']:.2f} & {row['saa']:.2f} & "
            f"{row['vza']:.2f} & {row['vaa']:.2f} & "
            f"{int(row['p_cross_m'])} & {int(row['p_along_m'])} \\\\"
        )

    lines.append(r'\bottomrule')
    lines.append(
        r'\multicolumn{10}{l}{%' + '\n'
                                   r'\vspace{4pt}' + '\n'
                                                     r'\parbox{\linewidth}{\scriptsize '
                                                     r'SZA: solar zenith angle; SAA: solar azimuth angle; '
                                                     r'VZA: satellite viewing zenith angle; VAA: satellite viewing azimuth angle; '
                                                     r'$p_\times$: cross-track pixel size; $p_\parallel$: along-track pixel size. '
                                                     r'MODIS and VIIRS corrected for Earth curvature; Sentinel-2B flat-Earth approximation.}}\\'
    )
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')
    if clearpage:
        lines.append(r'\clearpage')
    # lines.append(r'\end{landscape}')
    return '\n'.join(lines)



def generate_latex_figures(
    fig_path,
    output_tex_path,
    caption=None,
    section_title="Figures",
    is_landscape=False,
    use_appendix=False,
    overwrite=True,
    width_portrait="\\textwidth",
    width_landscape="\\textheight",
):
    """
    Automatically generate LaTeX figure blocks for all PNGs in fig_dir.
    """

    fig_path = Path(fig_path)
    output_tex_path = Path(output_tex_path)

    latex_lines = []

    # Optional appendix header
    if use_appendix:
        latex_lines.append(r"\clearpage")
        latex_lines.append(r"\appendix")
        latex_lines.append(rf"\section{{{section_title}}}")
        latex_lines.append("")

    filename = fig_path.stem  # terra_zoom_part1
    label = f"fig:{filename}"

    if is_landscape:
        latex_lines.append(r"\begin{landscape}")
        latex_lines.append(r"\begin{figure}[h!p]")
        latex_lines.append(r"\centering")
        latex_lines.append(
            rf"\includegraphics[width={width_landscape}]{{{fig_path.as_posix()}}}"
        )
    else:
        latex_lines.append(r"\begin{figure}[h!p]")
        latex_lines.append(r"\centering")
        latex_lines.append(
            rf"\includegraphics[width={width_portrait}]{{{fig_path.as_posix()}}}"
        )

    latex_lines.append(rf"\caption{{{{\footnotesize {caption}}}}}")
    latex_lines.append(rf"\label{{{label}}}")
    latex_lines.append(r"\end{figure}")

    if is_landscape:
        latex_lines.append(r"\end{landscape}")

    latex_lines.append(r"\clearpage")
    latex_lines.append("")

    # Write file
    mode = "w" if overwrite else "a"
    with open(output_tex_path, mode) as f:
        f.write("\n".join(latex_lines))

    # print(f"Generated LaTeX file: {output_tex_path}")
    return output_tex_path

def clean_satellite_name(nc_name):
    if nc_name.startswith("terra"):
        return "Terra MODIS"
    elif nc_name.startswith("noaa20"):
        return "NOAA-20 VIIRS"
    elif nc_name.startswith("sentinel2b"):
        return "Sentinel-2B MSI"
    else:
        return nc_name.split("_")[0].capitalize()


def format_class_name(name):
    return name.replace("_", " ").title()



def build_caption_piexel_selection(png_file, satfile_pixel_class_points):
    """
    Build caption using corrected satellite dictionary entries only.
    """

    stem = png_file.stem
    satname = stem.split("_")[0]
    class_lines =[]
    for nc_file in satfile_pixel_class_points:
        if not "uncorrected" in nc_file and stem.startswith(Path(nc_file).stem.split("_")[0]):
            for pixel_class, (lat, lon) in satfile_pixel_class_points[nc_file].items():
                         class_lines.append(
                            f"{format_class_name(pixel_class)} ({lat:.4f}°, {lon:.4f}°)")


    class_lines = sorted(class_lines)

    if "part1" in stem:
        part = ""
        cl = class_lines[0:5]

    if "part2" in stem:
        part = " continued."
        cl = class_lines[5:10]

    if 'all' in stem:
        part = ""
        cl = class_lines

    class_text = "; ".join(cl)

    caption = (
        f"{satname.capitalize()} true-color zoom analysis across surface and atmospheric classes{part}."
        f" Rows correspond to different land-cover and atmospheric classes, and columns show zoom levels with radii of 100, 40, and 12 pixels."
        f" The red outline highlights the selected pixel region for each class. Pixel class coordinates are as follows (degrees north, degrees east): "
        f" {class_text}."
    )

    return caption

In [5]:
def make_band_table_stdv(satname, cor_type='corrected'):
    """Single table per satellite combining reflective and emissive bands."""
    band_meta = SAT_BAND_METADATA.get(satname)
    if band_meta is None:
        print(f"No band metadata for {satname}")
        return None

    sat_data_ref = {}
    sat_data_bt = {}

    for (sn, ct), pixel_classes in spectral_analysis.items():
        if sn != satname or ct != cor_type:
            continue
        for pixel_class, sa in pixel_classes.items():
            sat_data_ref[pixel_class] = dict(zip(sa['wl_ref_sorted'], sa['reflectance_stdv']))
            sat_data_bt[pixel_class] = dict(zip(sa['wl_bt_sorted'], sa['brightness_temp_stdv']))
            print(sa['reflectance_mean'])
            print()
            print(sa['reflectance_stdv'])
    if not sat_data_ref and not sat_data_bt:
        print(f"No {cor_type} data found for {satname}")
        return None

    all_classes = sorted(set(list(sat_data_ref.keys()) + list(sat_data_bt.keys())))

    rows = []
    for band_key, info in band_meta.items():
        lo, hi = [float(x) for x in info['wavelength'].split('–')]
        wl_c = round((lo + hi) / 2, 4)
        is_emissive = lo >= 3.0
        band_type = 'Emissive' if is_emissive else 'Reflective'

        row = {
            'Band': band_key,
            'Wavelength': f"{lo:.3f}–{hi:.3f}",
            'Type': band_type,
        }
        for pixel_class in all_classes:
            data = sat_data_bt if is_emissive else sat_data_ref
            val = find_val(pixel_class, wl_c, data)
            row[clean_class(pixel_class)] = round(val, 3) if not np.isnan(val) else None

        rows.append(row)

    df = pd.DataFrame(rows)
    df['_lo'] = df['Wavelength'].apply(lambda x: float(x.split('–')[0]))
    df = (df[df['Type'] == 'Reflective'].sort_values('_lo')
          ._append(df[df['Type'] == 'Emissive'].sort_values('_lo'))
          .drop(columns='_lo')
          .reset_index(drop=True))
    return df

In [6]:

# ─── 1. Accumulation ──────────────────────────────────────────────────────────
spectral_analysis = {}

for file in all_data_paths:
    ds = xr.open_dataset(file)
    filename = file.name

    if 'uncorrected' in filename:
        cor_type = 'uncorrected'
    elif 'corrected' in filename:
        cor_type = 'corrected'
    else:
        raise ValueError(f"Cannot determine correction type from filename: {filename}")

    satname = filename.split("_")[0]
    sat_key = (satname, cor_type)

    for pixel_class, latlon in satfile_pixel_class_points[filename].items():
        pixel, (y, yy, x, xx) = pixel_selector(ds, lat_lon_point=latlon, radius=1)

        reflectance_bands, wl_ref, wl_ref_lower, wl_ref_upper = [], [], [], []
        bt_bands, wl_bt, wl_bt_lower, wl_bt_upper = [], [], [], []

        for var in pixel.data_vars:
            try:
                name = pixel[var].attrs['standard_name']
            except KeyError:
                continue

            if name == 'toa_bidirectional_reflectance':
                if np.isnan(pixel[var].values).all():
                    continue
                wl_parsed = parse_wavelength(pixel[var].attrs['wavelength'])
                reflectance_bands.append(var)
                wl_ref.append(wl_parsed['center'])
                wl_ref_lower.append(wl_parsed['lower'])
                wl_ref_upper.append(wl_parsed['upper'])
            elif name == 'toa_brightness_temperature':
                if np.isnan(pixel[var].values).all():
                    continue
                wl_parsed = parse_wavelength(pixel[var].attrs['wavelength'])
                bt_bands.append(var)
                wl_bt.append(wl_parsed['center'])
                wl_bt_lower.append(wl_parsed['lower'])
                wl_bt_upper.append(wl_parsed['upper'])

        if len(wl_ref) == 0 and len(wl_bt) == 0:
            print(f"Warning: no valid bands for {pixel_class} in {filename}, skipping.")
            continue

        wl_ref = np.array(wl_ref);
        wl_ref_lower = np.array(wl_ref_lower);
        wl_ref_upper = np.array(wl_ref_upper)
        wl_bt = np.array(wl_bt);
        wl_bt_lower = np.array(wl_bt_lower);
        wl_bt_upper = np.array(wl_bt_upper)

        ref_mean = np.array([pixel[b].mean(dim=("y", "x")).values for b in reflectance_bands])
        ref_stdv = np.array([pixel[b].std(dim=("y", "x"), ddof=0).values for b in reflectance_bands])
        bt_mean = np.array([pixel[b].mean(dim=("y", "x")).values for b in bt_bands])
        bt_stdv = np.array([pixel[b].std(dim=("y", "x"), ddof=0).values for b in bt_bands])

        ref_idx = np.argsort(wl_ref) if len(wl_ref) > 0 else np.array([], dtype=int)
        bt_idx = np.argsort(wl_bt) if len(wl_bt) > 0 else np.array([], dtype=int)

        spectral_analysis.setdefault(sat_key, {})[pixel_class] = {
            "reflectance_mean": ref_mean[ref_idx],
            "reflectance_stdv": ref_stdv[ref_idx],
            "brightness_temp_mean": bt_mean[bt_idx],
            "brightness_temp_stdv": bt_stdv[bt_idx],
            "wl_ref_sorted": wl_ref[ref_idx],
            "wl_ref_lower": wl_ref_lower[ref_idx],
            "wl_ref_upper": wl_ref_upper[ref_idx],
            "wl_bt_sorted": wl_bt[bt_idx],
            "wl_bt_lower": wl_bt_lower[bt_idx],
            "wl_bt_upper": wl_bt_upper[bt_idx],
        }

print(f"Accumulated {len(spectral_analysis)} sat/cor_type combinations")

Accumulated 6 sat/cor_type combinations


In [7]:
print(spectral_analysis[('noaa20', 'corrected')]['inland_water']['reflectance_mean'])
print()

print(spectral_analysis[('noaa20', 'corrected')]['inland_water']['reflectance_stdv'])



[1.8853773  2.125807   2.4302826  1.4165546  0.3792984  0.38684008
 1.443573   0.9096354  0.914955   0.32378507 0.07434537 0.17208228
 0.16355875 0.06204004]

[0.08316806 0.15159176 0.18589056 0.11371855 0.02180297 0.01253061
 0.00559242 0.00680978 0.00362897 0.01337099 0.01064749 0.01623652
 0.01407459 0.02277563]


In [8]:


# Plots all 10 classes !!!!


# from collections import defaultdict
#
# # Group files by satellite
# sat_files = defaultdict(list)
# for file in corrected_data_paths:
#     satname = file.name.split('_')[0]
#     sat_files[satname].append(file)
#
# radii = [100, 40, 12]
#
# for satname, files in sat_files.items():
#     # Collect all classes across files
#     all_classes = set()
#     for file in files:
#         all_classes.update(satfile_pixel_class_points[file.name].keys())
#     pixel_classes = sorted(list(all_classes))
#     n_classes = len(pixel_classes)
#     n_radii = len(radii)
#
#     fig, axes = plt.subplots(n_classes, n_radii, figsize=(n_radii*3, n_classes*3))
#     if n_classes == 1:
#         axes = axes[np.newaxis, :]
#     if n_radii == 1:
#         axes = axes[:, np.newaxis]
#
#     # Preload all datasets and masks for this satellite
#     loaded_data = []
#     for file in files:
#         ds = xr.open_dataset(file)
#         if satname == 'noaa20':
#             ds = ds.isel(y=slice(None, None, -1), x=slice(None, None, -1))
#         loaded_data.append((file.name, ds))
#
#     for row_idx, pixel_class in enumerate(pixel_classes):
#         for col_idx, radius in enumerate(radii):
#             ax = axes[row_idx, col_idx]
#             ax.axis('off')
#
#             # Overlay each file that has this class
#             for fname, ds in loaded_data:
#                 if pixel_class not in satfile_pixel_class_points[fname]:
#                     continue
#
#                 latlon = satfile_pixel_class_points[fname][pixel_class]
#
#                 # Base mask
#                 _, (ystart, yend, xstart, xend) = pixel_selector(ds, lat_lon_point=latlon, radius=1)
#                 class_mask = np.zeros(ds['true_color'].shape[1:3], dtype=bool)
#                 class_mask[ystart:yend, xstart:xend] = True
#
#                 pixel_zoom, (ys, ye, xs, xe) = pixel_selector(ds, lat_lon_point=latlon, radius=radius)
#                 rgb_zoom = get_enhanced_image(pixel_zoom['true_color']).data.transpose('y','x','bands').values
#                 zoom_mask = class_mask[ys:ye, xs:xe]
#
#                 # alpha_val = 1.0 if fname == loaded_data[0][0] else 0.3  # first file opaque, others transparent
#                 ax.imshow(rgb_zoom, origin='upper')
#                 ax.contour(zoom_mask.astype(float), levels=[0.5], colors='red', linewidths=1.5 if alpha_val==1 else 1.0)
#
#             # Column headers
#             if row_idx == 0:
#                 ax.set_title(f'r = {radius}', fontsize=10, fontweight='bold', pad=4)
#
#             # Row labels
#             if col_idx == 0:
#                 ax.text(
#                     -0.08, 0.5,
#                     clean_class(pixel_class),
#                     transform=ax.transAxes,
#                     fontsize=9, va='center', ha='right',
#                     rotation=90
#                 )
#
#     sat_clean = clean_satname(satname)
#     fig.suptitle(f'{sat_clean} · All Files Combined · Zoom Levels', fontsize=13, fontweight='bold', y=1.01)
#
#     plt.tight_layout(pad=0.5)
#     save_path = out_dir / f'{satname}_all_files_all_classes_zoom.png'
#     fig.savefig(save_path, dpi=200, bbox_inches='tight')
#     plt.show()
#     print(f'Saved {save_path.name}')

In [9]:
%%capture
# Group files by satellite
sat_files = defaultdict(list)
for file in corrected_data_paths:
    satname = file.name.split('_')[0]
    sat_files[satname].append(file)

radii = [100, 40, 12]
split_size = 5  # split after 5 classes

for satname, files in sat_files.items():

    # Collect all classes across files
    all_classes = set()
    for file in files:
        all_classes.update(satfile_pixel_class_points[file.name].keys())

    pixel_classes = sorted(list(all_classes))
    n_radii = len(radii)

    # Preload datasets once
    loaded_data = []
    for file in files:
        ds = xr.open_dataset(file)
        if satname == 'noaa20':
            ds = ds.isel(y=slice(None, None, -1), x=slice(None, None, -1))
        loaded_data.append((file.name, ds))

    # Split classes into chunks of 5
    class_chunks = [
        pixel_classes[i:i + split_size]
        for i in range(0, len(pixel_classes), split_size)
    ]

    for chunk_idx, class_subset in enumerate(class_chunks):

        n_classes = len(class_subset)

        fig, axes = plt.subplots(
            n_classes, n_radii,
            figsize=(n_radii*3, n_classes*3)
        )

        if n_classes == 1:
            axes = axes[np.newaxis, :]
        if n_radii == 1:
            axes = axes[:, np.newaxis]

        for row_idx, pixel_class in enumerate(class_subset):
            for col_idx, radius in enumerate(radii):

                ax = axes[row_idx, col_idx]
                ax.axis('off')

                for fname, ds in loaded_data:
                    if pixel_class not in satfile_pixel_class_points[fname]:
                        continue

                    latlon = satfile_pixel_class_points[fname][pixel_class]

                    # Base mask
                    _, (ystart, yend, xstart, xend) = pixel_selector(
                        ds, lat_lon_point=latlon, radius=1
                    )
                    class_mask = np.zeros(ds['true_color'].shape[1:3], dtype=bool)
                    class_mask[ystart:yend, xstart:xend] = True

                    pixel_zoom, (ys, ye, xs, xe) = pixel_selector(
                        ds, lat_lon_point=latlon, radius=radius
                    )
                    rgb_zoom = get_enhanced_image(
                        pixel_zoom['true_color']
                    ).data.transpose('y','x','bands').values

                    zoom_mask = class_mask[ys:ye, xs:xe]

                    ax.imshow(rgb_zoom, origin='upper')
                    ax.contour(
                        zoom_mask.astype(float),
                        levels=[0.5],
                        colors='red',
                        linewidths=1.5
                    )

                # Column headers
                if row_idx == 0:
                    ax.set_title(
                        f'r = {radius}',
                        fontsize=10,
                        fontweight='bold',
                        pad=4
                    )

                # Row labels
                if col_idx == 0:
                    ax.text(
                        -0.08, 0.5,
                        clean_class(pixel_class),
                        transform=ax.transAxes,
                        fontsize=9,
                        va='center',
                        ha='right',
                        rotation=90
                    )

        sat_clean = clean_satname(satname)

        fig.suptitle(
            f'{sat_clean} · Zoom Levels (Classes {chunk_idx*5+1}-{chunk_idx*5+len(class_subset)})',
            fontsize=13,
            fontweight='bold',
            y=1.01
        )

        plt.tight_layout(pad=0.5)

        save_path = out_dir / f'{satname}_zoom_part{chunk_idx+1}.png'
        fig.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.show()

        print(f'Saved {save_path.name}')

In [10]:
%%capture
# ─── 4. Spectral analysis plots ───────────────────────────────────────────────
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)

# Per-satellite plots
for (satname, cor_type), pixel_classes in spectral_analysis.items():
    fallback = sns.color_palette("tab10", n_colors=len(pixel_classes))
    fallback_iter = iter(fallback)
    colors = {pc: CLASS_COLORS.get(pc, next(fallback_iter)) for pc in pixel_classes}

    for band_type, x_key, y_key, yerr_key, x_lo, x_hi, ylabel, ylim, xlim in [
        ('Reflective', 'wl_ref_sorted', 'reflectance_mean', 'reflectance_stdv',
         'wl_ref_lower', 'wl_ref_upper', 'Reflectance (%)', (0, 140), (0.25, 2.3)),
        ('Emissive', 'wl_bt_sorted', 'brightness_temp_mean', 'brightness_temp_stdv',
         'wl_bt_lower', 'wl_bt_upper', 'Brightness Temperature (K)', (220, 340), (3, 15)),
    ]:
        fig, ax = plt.subplots(figsize=(11, 5))
        for pixel_class, sa in pixel_classes.items():
            plot_spectral_class(ax, sa[x_key], sa[y_key], sa[yerr_key],
                                # sa[x_lo], sa[x_hi],
                                label=clean_class(pixel_class),
                                color=colors[pixel_class],
                                )

        if band_type == 'Reflective':
            ax.axhspan(100, 140, color='#888888', alpha=0.12, zorder=0)
            ax.axhline(100, color='#888888', linewidth=0.8, linestyle='--', zorder=1)

        ax.set_ylim(*ylim);
        ax.set_xlim(*xlim);
        ax.margins(x=0)
        ax.set_title(f'{clean_satname(satname)}  ·  {band_type} Bands  ·  {clean_cortype(cor_type)}',
                     fontsize=13, fontweight='bold', pad=12)
        ax.set_xlabel('Wavelength (µm)', fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.,
                  framealpha=0.9, edgecolor='#cccccc', fontsize=9)
        sns.despine(ax=ax, left=False, bottom=False)
        fig.tight_layout()
        fig.savefig(out_dir / f'{satname.upper()}_{band_type}_{cor_type}.png', dpi=150, bbox_inches='tight')
        plt.show()

# Per-class plots
all_classes = set()
for pixel_classes in spectral_analysis.values():
    all_classes.update(pixel_classes.keys())

atmospheric = sorted(all_classes & ATMOSPHERIC_CLASSES)
land = sorted(all_classes & LAND_CLASSES)

print("=== ATMOSPHERIC CLASSES ===")
for pixel_class in atmospheric:
    make_class_plots(pixel_class, 'reflectance', outputdir=out_dir)
    make_class_plots(pixel_class, 'emissive', outputdir=out_dir)

print("=== LAND CLASSES ===")
for pixel_class in land:
    make_class_plots(pixel_class, 'reflectance', outputdir=out_dir)
    make_class_plots(pixel_class, 'emissive', outputdir=out_dir)


In [11]:

# ─── 5. Stats & CSVs ──────────────────────────────────────────────────────────
records = []
for (satname, cor_type), pixel_classes in spectral_analysis.items():
    for pixel_class, sa in pixel_classes.items():
        for i, wl in enumerate(sa['wl_ref_sorted']):
            records.append({
                'satellite': clean_satname(satname),
                'correction_type': clean_cortype(cor_type),
                'class': clean_class(pixel_class),
                'band_type': 'reflectance',
                'wavelength': round(wl, 4),
                'wl_lower': round(sa['wl_ref_lower'][i], 4),
                'wl_upper': round(sa['wl_ref_upper'][i], 4),
                'mean': round(sa['reflectance_mean'][i], 4),
                'stdv': round(sa['reflectance_stdv'][i], 4),
            })
        for i, wl in enumerate(sa['wl_bt_sorted']):
            records.append({
                'satellite': clean_satname(satname),
                'correction_type': clean_cortype(cor_type),
                'class': clean_class(pixel_class),
                'band_type': 'brightness_temp',
                'wavelength': round(wl, 4),
                'wl_lower': round(sa['wl_bt_lower'][i], 4),
                'wl_upper': round(sa['wl_bt_upper'][i], 4),
                'mean': round(sa['brightness_temp_mean'][i], 4),
                'stdv': round(sa['brightness_temp_stdv'][i], 4),
            })

df_stats = pd.DataFrame(records)
df_stats.to_csv(stats_dir / 'spectral_stats_per_band.csv', index=False)
print(df_stats['band_type'].value_counts())

df_ref_summary = make_summary_with_pctdiff(df_stats, 'reflectance')
df_bt_summary = make_summary_with_pctdiff(df_stats, 'brightness_temp')
df_ref_summary.to_csv(stats_dir / 'reflectance_summary.csv', index=False)
df_bt_summary.to_csv(stats_dir / 'brightness_temp_summary.csv', index=False)


band_type
reflectance        960
brightness_temp    460
Name: count, dtype: int64


In [12]:
# ─── 6. LaTeX summary tables ──────────────────────────────────────────────────
latex_ref = summary_to_latex(
    df_ref_summary,
    caption='Mean TOA reflectance (\\%) by class and satellite. '
            'Percent difference between corrected and uncorrected; values $>$5\\% highlighted.',
    label='tab:reflectance_summary'
)
latex_bt = summary_to_latex(
    df_bt_summary,
    caption='Mean brightness temperature (K) by class and satellite. '
            'Atmospheric correction does not modify emissive band radiances.',
    label='tab:bt_summary'
)
combined = latex_ref + '\n\n' + latex_bt

for fname, content in [
    ('table_reflectance.tex', latex_ref),
    ('table_brightness_temp.tex', latex_bt),
    ('all_tables.tex', combined),
]:
    (stats_dir / fname).write_text(content)
    shutil.copy(stats_dir / fname, latex_dir / fname)
    print(f'Saved {fname}')


Saved table_reflectance.tex
Saved table_brightness_temp.tex
Saved all_tables.tex


In [13]:

# ─── 7. Band tables per satellite ─────────────────────────────────────────────
all_sats = sorted(set(sn for (sn, _) in spectral_analysis.keys()))

for satname in all_sats:
    sat_clean = clean_satname(satname)

    for cor_type, suffix, ct_label in [
        ('corrected', 'corrected', 'Corrected'),
        ('uncorrected', 'uncorrected', 'Uncorrected'),
    ]:
        df_band = make_band_table(satname, cor_type)
        if df_band is None or df_band.empty:
            continue
        caption = (f'{sat_clean} band wavelengths and mean spectral response per class '
                   f'({ct_label}). Reflective in \\%; emissive in K.')
        label = f'tab:{satname}_bands_{suffix}'
        fname = f'table_{satname}_bands_{suffix}.tex'
        latex = band_df_to_latex(df_band, satname, caption, label)
        (stats_dir / fname).write_text(latex)
        shutil.copy(stats_dir / fname, latex_dir / fname)
        print(f'Saved {fname}')

        df_band_stdv = make_band_table_stdv(satname, cor_type)
        if df_band is None or df_band.empty:
            continue
        caption = (f'{sat_clean} band wavelengths and class-wise standard deviation of spectral response.'
                   f'({ct_label}). Reflective in \\%; emissive in K.')
        label = f'tab:{satname}_bands_stdv_{suffix}'
        fname = f'table_{satname}_bands_stdv_{suffix}.tex'
        latex = band_df_to_latex(df_band_stdv, satname, caption, label, is_stdv=True)
        (stats_dir / fname).write_text(latex)
        shutil.copy(stats_dir / fname, latex_dir / fname)
        print(f'Saved {fname}')

    df_pct = make_band_table_pctdiff(satname)
    if df_pct is not None and not df_pct.empty:
        caption = (f'{sat_clean} percent difference in spectral response between '
                   f'corrected and uncorrected (reflective bands). '
                   f'Values $>$20\\% highlighted.')
        label = f'tab:{satname}_bands_pctdiff'
        fname = f'table_{satname}_bands_pctdiff.tex'
        latex = band_df_to_latex(df_pct, satname, caption, label, is_pctdiff=True)
        (stats_dir / fname).write_text(latex)
        shutil.copy(stats_dir / fname, latex_dir / fname)
        print(f'Saved {fname}')

Saved table_noaa20_bands_corrected.tex
[1.8853773  2.125807   2.4302826  1.4165546  0.3792984  0.38684008
 1.443573   0.9096354  0.914955   0.32378507 0.07434537 0.17208228
 0.16355875 0.06204004]

[0.08316806 0.15159176 0.18589056 0.11371855 0.02180297 0.01253061
 0.00559242 0.00680978 0.00362897 0.01337099 0.01064749 0.01623652
 0.01407459 0.02277563]
[94.739365  97.22019   98.7085    96.264946  95.69711   97.969376
 10.4437065 88.40492   88.59328   46.13099    7.989546   8.615305
  8.697399  12.49365  ]

[7.724161   8.208262   8.644943   8.705911   9.028594   9.163472
 0.27258626 8.559844   8.420568   4.653808   0.7888231  0.9486933
 0.88916695 1.243079  ]
[ 7.958788   10.894328   13.696518   20.042053   25.12044    26.823553
 12.118381   30.643127   30.688942   34.538418    0.11206841 39.57028
 40.182713   37.2118    ]

[0.09250222 0.11154205 0.12996957 0.18255106 0.26183853 0.25055367
 0.04549686 0.31543487 0.31497225 0.35074008 0.01081803 0.36599717
 0.32313886 0.24108985]
[ 3.06

In [14]:
# ─── 8. Pixel info table ──────────────────────────────────────────────────────
pixel_info_records = []

for file in all_data_paths:
    ds = xr.open_dataset(file)
    filename = file.name

    if 'uncorrected' in filename:
        cor_type = 'uncorrected'
    elif 'corrected' in filename:
        cor_type = 'corrected'
    else:
        continue

    if cor_type != 'corrected':
        continue

    satname = filename.split('_')[0]
    params = SAT_ORBIT_PARAMS.get(satname)
    if params is None:
        continue

    for pixel_class, latlon in satfile_pixel_class_points[filename].items():
        try:
            info = get_pixel_info(
                sat_data=ds,
                nadir_along_track_resolution=params['along_track_res'],
                nadir_cross_track_resoltuion=params['cross_track_res'],
                sat_orb_height=params['orbit_height'],
                correct_for_earth_curvature=params['earth_curvature'],
                lat_lon_point=latlon,
                radius=1
            )
        except Exception as e:
            print(f"  WARNING: {satname} | {pixel_class} | {e}")
            continue

        _, (ys, ye, xs, xe) = pixel_selector(ds, lat_lon_point=latlon, radius=1)
        cy = (ys + ye) // 2
        cx = (xs + xe) // 2

        sza_c = float(ds['solar_zenith_angle'].values[cy, cx])
        saa_c = float(ds['solar_azimuth_angle'].values[cy, cx])
        vza_c = float(ds['satellite_zenith_angle'].values[cy, cx])
        vaa_c = float(ds['satellite_azimuth_angle'].values[cy, cx])
        lat_c = float(ds['latitude'].values[cy, cx])
        lon_c = float(ds['longitude'].values[cy, cx])

        cy_loc = info['pixel_size_crosstrack'].shape[0] // 2
        cx_loc = info['pixel_size_crosstrack'].shape[1] // 2

        p_cross_center = float(info['pixel_size_crosstrack'][cy_loc, cx_loc])
        p_along_center = float(info['pixel_size_along_track'][cy_loc, cx_loc])
        p_cross_nadir = float(np.nanmean(info['pixel_size_crosstrack_nadir']))
        p_along_nadir = float(np.nanmean(info['pixel_size_along_track_nadir']))
        p_eff = info['effective_pixel_size']
        p_eff_mean = float(info['mean_effective_pixel_size'])
        p_eff_stdv= float(info['stdv_effective_pixel_size'])

        pixel_info_records.append({
            'satellite': params['clean_name'],
            'class': clean_class(pixel_class),
            'latitude': round(lat_c, 4),
            'longitude': round(lon_c, 4),
            'sza': round(sza_c, 2),
            'saa': round(saa_c, 2),
            'vza': round(vza_c, 2),
            'vaa': round(vaa_c, 2),
            'p_cross_m': round(p_cross_center),
            'p_along_m': round(p_along_center),
            'p_cross_nadir_m': round(p_cross_nadir),
            'p_along_nadir_m': round(p_along_nadir),
            'p_eff_mean_m': round(p_eff_mean),
            'p_eff_stdv_m': round(p_eff_stdv),
        })

df_pixel_info = pd.DataFrame(pixel_info_records)
df_pixel_info.to_csv(stats_dir / 'pixel_info.csv', index=False)

latex_pixel = pixel_info_to_latex(
    df_pixel_info,
    caption='Viewing geometry and pixel size at center pixel for each satellite and land/atmosphere class.',
    label='tab:pixel_info'
)
(stats_dir / 'table_pixel_info.tex').write_text(latex_pixel)
shutil.copy(stats_dir / 'table_pixel_info.tex', latex_dir / 'table_pixel_info.tex')
print('Saved table_pixel_info.tex')


Saved table_pixel_info.tex


In [15]:
png_plots = Path('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/pixel_selection_plots/png_plots').glob("*.png")
outputdir = Path('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/pixel_selection_plots')

for i, file in enumerate(sorted(png_plots)):

    cap = build_caption_piexel_selection(file, satfile_pixel_class_points)
    satname = file.stem.split("_")[0]
    part = file.stem.split("_")[-1]
    generate_latex_figures(
        fig_path=file,
        output_tex_path= outputdir / f"pixel_selection_{satname}_{part}.tex",
        caption=cap,
        section_title="Figures",
        is_landscape=False,
        use_appendix=False,
        overwrite=True,
        width_portrait="0.6\\textwidth",
        width_landscape="\\textheight",
                               )
    # print(cap)
    # print()

In [16]:
def clean_label(text):
    """Convert filename to safe LaTeX label."""
    text = str(text)
    text = text.lower()
    text = re.sub(r'\.tex$', '', text)
    text = re.sub(r'[^a-z0-9]+', '-', text)
    return text.strip('-')

def pretty_title(filename):
    """Turn filename into readable subsection title."""
    print(filename)
    name = os.path.basename(filename)
    name = name.replace('.tex', '')
    name = name.replace('_', ' ')
    print(name)
    print(name.title())
    return name.title()


def build_spectral_analysis_appendix_latex(filelist, output_tex_path, overwrite=True):
    lines = []

    # lines.append(r'\clearpage')
    # lines.append(r'\appendix')
    # lines.append(r'\renewcommand{\thesubsection}{\thesection\arabic{subsection}}')
    # lines.append('')
    # lines.append(r'\section{Tables}')
    # lines.append('')
    filelist = [f for f in filelist if "appendix" not in str(f).lower()]

    for file in filelist:

        title = pretty_title(file)

        label = clean_label(file)

        lines.append(rf'\subsection{{{title}}}')
        lines.append(rf'\label{{app:{label}}}')
        lines.append(rf'\input{{{file}}}')
        lines.append('')

        # Write file
        mode = "w" if overwrite else "a"
        with open(output_tex_path, mode) as f:
            f.write("\n".join(lines))

        print(f"Generated LaTeX file: {output_tex_path}")
    # return "\n".join(lines)

SATELLITE_MAP = {
    "NOAA20": "NOAA-20 VIIRS",
    "SENTINEL2B": "Sentinel-2B MSI",
    "TERRA": "Terra MODIS"
}

CLASS_MAP = {
    "dense_forest": "dense forest",
    "high_cloud": "high cloud",
    "inland_water": "inland water",
    "grassland": "grassland",
    "agriculture": "agriculture",
    "urban": "urban",
    "smoke": "smoke",
    "dust": "dust",
    "snow": "snow",
    "desert": "desert"
}

def generate_spectral_anlaysis_caption_latex(filename):
    name = os.path.basename(filename).replace(".png", "")
    parts = name.split("_")

    # --- Case 1: Class across satellites ---
    if parts[-1] == "satellites":

        class_key = "_".join(parts[:-3])  # handles dense_forest etc.

        band_type = parts[-3]
        class_name = CLASS_MAP.get(class_key, class_key.replace("_", " "))

        return (
            f"{str(band_type).capitalize()} band for {class_name} class spectral signatures  "
            f"across all satellites (NOAA-20 VIIRS, Sentinel-2B MSI, and Terra MODIS)."
        )

    # --- Case 2: Single satellite corrected/uncorrected ---
    else:
        satellite = parts[0]
        band_type = parts[1]
        processing = parts[2]
        sat_name = SATELLITE_MAP.get(satellite, satellite)
        band_type = band_type.lower()

        if processing == "corrected":
            return (
                f"{band_type.capitalize()} band spectral response for {sat_name} "
                f"after radiometric correction."
            )
        else:
            return (
                f"{band_type.capitalize()} band spectral response for {sat_name} "
                f"prior to radiometric correction."
            )

In [51]:
png_plots = Path('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/png_plots').glob("*.png")
outputdir = Path('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots')

for i, file in enumerate(sorted(png_plots)):

    cap = generate_spectral_anlaysis_caption_latex(file)
    satname = file.stem.split("_")[0]
    satvalue =[]
    part = []
    print(file)
    if satname in SATELLITE_MAP:
        part = file.stem.split("_")[-2:]
        part = "_".join(part)

    elif satname in CLASS_MAP:
        sat_value = CLASS_MAP[satname]
        part = file.stem.split("_")[-3:-1]
        part = "_".join(part)
    else:
        # If we found a match (exact or prefix), join if it's a list/tuple
        satname = "_".join(file.stem.split("_")[0:2])
        part = "_".join(file.stem.split("_")[2:])
        # else:
        #     # 4. Fallback: Parse from filename if no map match exists
        #     # Standardizing to the first two parts of the filename
        #     satname = "_".join(file.stem.split("_")[:2])
    #
    # if satname in CLASS_MAP:
    #     part = file.stem.split("_")[-2:]
    #     part = "_".join(part)

    print('satname - ', satname, '      part - ', part)
    generate_latex_figures(
        fig_path=file,
        output_tex_path= outputdir / f"spectral_response_{satname}_{part}.tex",
        caption=cap,
        section_title="Figures",
        is_landscape=False,
        use_appendix=False,
        overwrite=True,
        width_portrait="\\textwidth",
        width_landscape="\\textheight",
                               )

filelist = (sorted(outputdir.glob("*NOAA*Reflective*corrected*.tex"))
            + sorted(outputdir.glob("*NOAA*Emissive*corrected*.tex"))
            + sorted(outputdir.glob("*SENT*Reflective*corrected*.tex"))
            # + sorted(outputdir.glob("*SENT*Emissive*corrected*.tex"))
            + sorted(outputdir.glob("*TERRA*Reflective*corrected*.tex"))
            + sorted(outputdir.glob("*TERRA*Emissive*corrected*.tex"))
            + sorted(outputdir.glob("*Reflective*all*.tex"))
            + sorted(outputdir.glob("*Emissive*all*.tex"))
            )

build_spectral_analysis_appendix_latex(filelist, outputdir / f"spectral_response_appendix.tex")



/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/png_plots/NOAA20_Emissive_corrected.png
satname -  NOAA20       part -  Emissive_corrected
/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/png_plots/NOAA20_Emissive_uncorrected.png
satname -  NOAA20       part -  Emissive_uncorrected
/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/png_plots/NOAA20_Reflective_corrected.png
satname -  NOAA20       part -  Reflective_corrected
/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/png_plots/NOAA20_Reflective_uncorrected.png
satname -  NOAA20       part -  Reflective_uncorrected
/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/png_plots/SENTINEL2B_Emissive_corrected.png
satname -  SENTINEL2B       part -  Emissive_corrected
/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/png_plots/SENTINEL2B_Emissive_uncorrected.png
satname -  SE

In [50]:
sorted(outputdir.glob("*Emissive*all*.tex"))

[PosixPath('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/spectral_response_agriculture_Emissive_all.tex'),
 PosixPath('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/spectral_response_dense_forest_Emissive_all_satellites.tex'),
 PosixPath('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/spectral_response_desert_Emissive_all.tex'),
 PosixPath('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/spectral_response_dust_Emissive_all.tex'),
 PosixPath('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/spectral_response_grassland_Emissive_all.tex'),
 PosixPath('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/spectral_response_high_cloud_Emissive_all_satellites.tex'),
 PosixPath('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/spectral_response_inland_water_Emissive_all_satellites.tex'),
 PosixP

In [18]:
png_plots = Path('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/png_plots').glob("*.png")
testfile = sorted(Path('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/png_plots').glob("*high_cloud*.png"))[0]

In [19]:
png_plots = Path('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/png_plots').glob("*.png")

for i, file in enumerate(sorted(png_plots)):

    satname = file.stem.split("_")[0]
    part = file.stem.split("_")
    print(part)

['NOAA20', 'Emissive', 'corrected']
['NOAA20', 'Emissive', 'uncorrected']
['NOAA20', 'Reflective', 'corrected']
['NOAA20', 'Reflective', 'uncorrected']
['SENTINEL2B', 'Emissive', 'corrected']
['SENTINEL2B', 'Emissive', 'uncorrected']
['SENTINEL2B', 'Reflective', 'corrected']
['SENTINEL2B', 'Reflective', 'uncorrected']
['TERRA', 'Emissive', 'corrected']
['TERRA', 'Emissive', 'uncorrected']
['TERRA', 'Reflective', 'corrected']
['TERRA', 'Reflective', 'uncorrected']
['agriculture', 'Emissive', 'all', 'satellites']
['agriculture', 'Reflective', 'all', 'satellites']
['dense', 'forest', 'Emissive', 'all', 'satellites']
['dense', 'forest', 'Reflective', 'all', 'satellites']
['desert', 'Emissive', 'all', 'satellites']
['desert', 'Reflective', 'all', 'satellites']
['dust', 'Emissive', 'all', 'satellites']
['dust', 'Reflective', 'all', 'satellites']
['grassland', 'Emissive', 'all', 'satellites']
['grassland', 'Reflective', 'all', 'satellites']
['high', 'cloud', 'Emissive', 'all', 'satellites']
[

In [33]:
png_plots = Path('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/png_plots').glob("*Emissive*all*.png")
sorted(png_plots)

[PosixPath('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/png_plots/agriculture_Emissive_all_satellites.png'),
 PosixPath('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/png_plots/dense_forest_Emissive_all_satellites.png'),
 PosixPath('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/png_plots/desert_Emissive_all_satellites.png'),
 PosixPath('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/png_plots/dust_Emissive_all_satellites.png'),
 PosixPath('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/png_plots/grassland_Emissive_all_satellites.png'),
 PosixPath('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/png_plots/high_cloud_Emissive_all_satellites.png'),
 PosixPath('/Users/cwelch/Downloads/examples-master 2/Latex/SARS/spectral_response_plots/png_plots/inland_water_Emissive_all_satellites.png'),
 PosixPath('/Users/

In [27]:

spectral_analysis[('noaa20', 'corrected')]['inland_water']

{'reflectance_mean': array([1.8853773 , 2.125807  , 2.4302826 , 1.4165546 , 0.3792984 ,
        0.38684008, 1.443573  , 0.9096354 , 0.914955  , 0.32378507,
        0.07434537, 0.17208228, 0.16355875, 0.06204004], dtype=float32),
 'reflectance_stdv': array([0.08316806, 0.15159176, 0.18589056, 0.11371855, 0.02180297,
        0.01253061, 0.00559242, 0.00680978, 0.00362897, 0.01337099,
        0.01064749, 0.01623652, 0.01407459, 0.02277563], dtype=float32),
 'brightness_temp_mean': array([283.75829592, 283.34846024, 281.07355323, 283.01304012,
        285.43498937, 285.41180819, 285.5102679 ]),
 'brightness_temp_stdv': array([0.20842694, 0.25940305, 0.21993669, 0.17458399, 0.20177956,
        0.17199958, 0.1995812 ]),
 'wl_ref_sorted': array([0.412, 0.445, 0.488, 0.555, 0.64 , 0.672, 0.746, 0.865, 0.865,
        1.24 , 1.378, 1.61 , 1.61 , 2.25 ]),
 'wl_ref_lower': array([0.402, 0.436, 0.478, 0.545, 0.6  , 0.662, 0.739, 0.845, 0.846,
        1.23 , 1.371, 1.58 , 1.58 , 2.225]),
 'wl_ref_up

In [22]:
np.array([283.348, 283.758, 281.074, 283.013,285.435,285.412,285.510]).std()

np.float64(1.5270631677724824)

In [23]:
df_pixel_info

,satellite,class,latitude,longitude,sza,saa,vza,vaa,p_cross_m,p_along_m,p_cross_nadir_m,p_along_nadir_m,p_eff_mean_m,p_eff_stdv_m
0,NOAA 20 VIIRS,Inland Water,44.6090,81.1329,30.48,-148.76,14.69,-99.57,1027,775,993,750,1027,2
1,NOAA 20 VIIRS,Snow,42.1969,80.2406,28.02,-148.39,4.52,-97.63,775,752,773,750,776,1
2,NOAA 20 VIIRS,Desert,39.9531,80.2721,26.08,-146.25,0.13,108.62,750,750,750,750,750,0
3,NOAA 20 VIIRS,Agriculture,40.9926,80.2497,26.97,-147.29,2.03,-91.61,755,750,755,750,755,0
4,NOAA 20 VIIRS,Urban,43.8963,81.3302,29.93,-147.85,14.44,-100.17,1017,774,985,750,1018,2
5,NOAA 20 VIIRS,Grassland,42.8933,80.7822,28.84,-148.01,9.24,-103.02,857,760,846,750,858,1
6,NOAA 20 VIIRS,Dense Forest,43.0945,82.2410,29.59,-145.61,18.22,-101.14,1183,790,1124,750,1185,2
7,NOAA 20 VIIRS,Dust,16.5826,16.5358,16.16,-91.35,18.27,81.91,1186,790,1126,750,1187,3
8,NOAA 20 VIIRS,Smoke,62.1483,-116.1399,41.29,-178.39,5.68,68.71,790,754,786,750,790,1
9,NOAA 20 VIIRS,High Cloud,63.7735,-114.5348,42.95,-176.11,5.26,-108.74,784,753,781,750,784,1
